In [9]:
import csv
from pathlib import Path
import numpy as np
import pandas as pd

# Create data folder
DATA = Path("data") / "delivery_times.csv"

# Dataset settings
SEED = 42
N_ROWS = 600

# Create dataset if it does not exist
if not DATA.exists():

    DATA.parent.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(SEED)

    distance_km = np.round(
        rng.uniform(0.5, 12.0, N_ROWS), 2
    )

    prep_time_min = np.round(
        rng.uniform(5, 30, N_ROWS), 0
    )

    traffic_level = rng.integers(
        1, 4, N_ROWS
    )

    rain = rng.binomial(
        1, 0.25, N_ROWS
    )

    delivery_min = np.round(
        6.0
        + 3.1 * distance_km
        + 0.65 * prep_time_min
        + 4.2 * traffic_level
        + 5.5 * rain
        + rng.normal(0, 2.5, N_ROWS),
        1
    )

    with DATA.open("w", newline="", encoding="utf-8") as fh:

        writer = csv.writer(fh)

        writer.writerow([
            "distance_km",
            "prep_time_min",
            "traffic_level",
            "rain",
            "delivery_min"
        ])

        for i in range(N_ROWS):
            writer.writerow([
                distance_km[i],
                int(prep_time_min[i]),
                int(traffic_level[i]),
                int(rain[i]),
                delivery_min[i]
            ])

    print("Dataset created:", DATA)

else:
    print("Dataset already exists:", DATA)


# Load dataset
orders = pd.read_csv(DATA)

# Features and target
FEATURES = [
    "distance_km",
    "prep_time_min",
    "traffic_level",
    "rain"
]

X = orders[FEATURES]
y = orders["delivery_min"]


# Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Dataset shape:", orders.shape)
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Dataset created: data/delivery_times.csv
Dataset shape: (600, 5)
Training data: (480, 4)
Testing data: (120, 4)


In [10]:
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score

In [11]:
deep_tree = DecisionTreeRegressor(
    max_depth=12,
    random_state=42
)

deep_tree.fit(X_train, y_train)

T1_train_mae = mean_absolute_error(
    y_train,
    deep_tree.predict(X_train)
)

T1_test_mae = mean_absolute_error(
    y_test,
    deep_tree.predict(X_test)
)

T1_gap = T1_test_mae - T1_train_mae

print(
    f"train {T1_train_mae} "
    f"test {T1_test_mae} "
    f"gap {T1_gap}"
)

train 0.04695601851851851 test 3.397986111111111 gap 3.3510300925925924


In [12]:
T2_depths = [2, 4, 6, None]

T2_scores = {}

for depth in T2_depths:

    forest = RandomForestRegressor(
        n_estimators=50,
        max_depth=depth,
        random_state=42
    )

    mae = -cross_val_score(
        forest,
        X,
        y,
        cv=5,
        scoring="neg_mean_absolute_error"
    ).mean()

    T2_scores[depth] = mae


T2_best_depth = min(
    T2_scores,
    key=T2_scores.get
)

print("T2 scores:", T2_scores)
print("Best depth:", T2_best_depth)

T2 scores: {2: np.float64(5.271498260221752), 4: np.float64(3.6428592874080996), 6: np.float64(2.8455304615596027), None: np.float64(2.6852499999999986)}
Best depth: None


In [13]:
def compare(models):
    results = {}

    for name, model in models.items():

        # Train the model
        model.fit(X_train, y_train)

        # Make predictions
        predictions = model.predict(X_test)

        # Calculate MAE
        mae = mean_absolute_error(y_test, predictions)

        # Store the result
        results[name] = mae

    return results


T3_table = compare({
    "linear": LinearRegression(),
    "tree4": DecisionTreeRegressor(
        max_depth=4,
        random_state=42
    ),
    "forest": RandomForestRegressor(
        n_estimators=50,
        random_state=42
    ),
})

print(T3_table)

{'linear': 1.924653079833835, 'tree4': 4.228332168713071, 'forest': 2.394216666666664}
